# ST2 — DBT Gold Setup : RFM + CLV + Fidélisation (Google Colab)
**Projet RFM CLV E-Commerce**

Ce notebook **complète ST1** en ajoutant :
1. `mart_rfm`  — scores R, F, M + segment nommé par client
2. `mart_clv`  — Customer Lifetime Value simplifiée par segment
3. `mart_cohorts` — rétention mensuelle par cohorte d'acquisition
4. Tests dbt ST2 dans schema.yml
5. Vérification KPIs ST2 depuis DuckDB

> **Prérequis :** Le notebook ST1 doit avoir été exécuté en premier.
> `fact_transactions`, `dim_customer`, `dim_date` sont déjà dans gold.duckdb.

## Cellule 1 — Monter Google Drive (identique ST1)

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ============================================================
# ADAPTEZ CE CHEMIN — même valeur que dans ST1
# ============================================================
PROJECT_PATH = "/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce"
DBT_PATH     = f"{PROJECT_PATH}/DBT_Gold"
DATA_PATH    = f"{PROJECT_PATH}/Data"
SILVER_PATH  = f"{DATA_PATH}/silver"
GOLD_DB      = f"{DATA_PATH}/gold/gold.duckdb"

# Vérification
assert os.path.exists(PROJECT_PATH), f"Projet introuvable : {PROJECT_PATH}"
assert os.path.exists(SILVER_PATH),  f"Silver introuvable : {SILVER_PATH}"
assert os.path.exists(GOLD_DB),      f"gold.duckdb introuvable — lancez ST1 d'abord !"

print(' Drive monté')
print('   Projet   :', PROJECT_PATH)
print('   Silver   :', os.listdir(SILVER_PATH))
print('   Gold DB  :', GOLD_DB)

Mounted at /content/drive
 Drive monté
   Projet   : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce
   Silver   : ['ecommerce_shipping_silver.parquet', 'fact_returns.parquet', 'online_retail_silver.parquet']
   Gold DB  : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb


## Cellule 2 — Installer dbt-duckdb
> À relancer à chaque session Colab.

In [2]:
!pip install dbt-core dbt-duckdb duckdb --quiet
!dbt --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.7/442.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.7/238.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## Cellule 3 — Recréer profiles.yml
> Toujours dans `~/.dbt/` — éphémère en Colab.

In [3]:
import os

profiles_dir = os.path.expanduser('~/.dbt')
os.makedirs(profiles_dir, exist_ok=True)

profiles_content = f"""ecommerce:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: "{GOLD_DB}"
      threads: 4
"""

with open(f"{profiles_dir}/profiles.yml", 'w') as f:
    f.write(profiles_content)

print('profiles.yml recréé →', profiles_dir)
print(f'   DuckDB : {GOLD_DB}')

profiles.yml recréé → /root/.dbt
   DuckDB : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb


## Cellule 4 — Vérifier les tables ST1 déjà en place

In [4]:
import duckdb
import pandas as pd

con = duckdb.connect(GOLD_DB)

print('Tables déjà disponibles dans gold.duckdb :')
tables = con.execute('SHOW TABLES').df()
print(tables.to_string())

# Vérifier que fact_transactions existe (ST1 prérequis)
required = ['fact_transactions', 'dim_customer', 'dim_date']
existing = tables['name'].tolist() if 'name' in tables.columns else []

print('\nVérification prérequis ST1 :')
for t in required:
    status = '' if t in existing else ' MANQUANT — lancez ST1 d abord'
    print(f'  {status}  {t}')

con.close()

Tables déjà disponibles dans gold.duckdb :
                 name
0        dim_customer
1            dim_date
2         dim_payment
3        fact_returns
4   fact_transactions
5            mart_clv
6        mart_cohorts
7         mart_funnel
8            mart_rfm
9         stg_returns
10       stg_shipping
11   stg_transactions

Vérification prérequis ST1 :
    fact_transactions
    dim_customer
    dim_date


## Cellule 5 — Créer les dossiers ST2 dans DBT_Gold

In [5]:
# Les dossiers staging/ et marts/ existent déjà (ST1)
# On crée juste le sous-dossier ST2 pour l'organisation
folders = [
    'models/staging',
    'models/marts',
    'tests'
]
for folder in folders:
    os.makedirs(f"{DBT_PATH}/{folder}", exist_ok=True)

print('Structure DBT_Gold vérifiée')
for root, dirs, files in os.walk(DBT_PATH):
    level = root.replace(DBT_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

Structure DBT_Gold vérifiée
DBT_Gold/
  dbt_project.yml
  target/
    catalog.json
    index.html
    partial_parse.msgpack
    semantic_manifest.json
    run_results.json
    manifest.json
    graph_summary.json
    graph.gpickle
    run/
      ecommerce/
        models/
          staging/
            stg_returns.sql
            stg_shipping.sql
            stg_transactions.sql
          schema.yml/
            accepted_values_mart_rfm_m_score__1__2__3__4__5.sql
            accepted_values_mart_clv_99b1edfd532f102639ae3e8c36294d12.sql
            accepted_values_mart_rfm_f_score__1__2__3__4__5.sql
            accepted_values_mart_rfm_r_score__1__2__3__4__5.sql
            not_null_mart_clv_clv_simple.sql
            accepted_values_mart_rfm_471de7decfe4044180ad21535fd957f4.sql
            not_null_mart_cohorts_cohort_month.sql
            not_null_mart_clv_rfm_segment.sql
            not_null_mart_clv_customer_id.sql
            not_null_mart_cohorts_period_number.sql
            not_

## Cellule 6 — Écrire mart_rfm.sql
> Calcul des scores R, F, M (1→5) et affectation du segment nommé.

In [6]:
mart_rfm_sql = """
-- ============================================================
-- mart_rfm : Segmentation RFM — ST2
-- 1 ligne = 1 client avec scores R, F, M et segment nommé
-- ============================================================

with reference_date as (
    -- Date de référence = lendemain de la dernière transaction
    select max(invoice_date) + interval '1 day' as ref_date
    from fact_transactions
),

rfm_raw as (
    -- Calcul des 3 métriques brutes par client
    select
        t.customer_id,
        c.country,
        -- Récence : nb de jours depuis le dernier achat
        datediff('day', max(t.invoice_date), r.ref_date)   as recency_days,
        -- Fréquence : nb de factures distinctes
        count(distinct t.invoice_no)                        as frequency,
        -- Montant : CA total généré
        round(sum(t.line_revenue), 2)                       as monetary
    from fact_transactions t
    left join dim_customer  c using (customer_id)
    cross join reference_date r
    group by t.customer_id, c.country, r.ref_date
),

rfm_scores as (
    -- Scoring 1→5 par quintile (5 = meilleur)
    select
        customer_id,
        country,
        recency_days,
        frequency,
        monetary,
        -- R : score inversé (récent = 5)
        ntile(5) over (order by recency_days desc)   as r_score,
        -- F : score direct (fréquent = 5)
        ntile(5) over (order by frequency asc)       as f_score,
        -- M : score direct (gros CA = 5)
        ntile(5) over (order by monetary asc)        as m_score
    from rfm_raw
),

rfm_segmented as (
    select
        customer_id,
        country,
        recency_days,
        frequency,
        monetary,
        r_score,
        f_score,
        m_score,
        -- Score composite RFM
        round((r_score + f_score + m_score) / 3.0, 2)  as rfm_score,
        -- Segmentation nommée selon grille CDC
        case
            when r_score >= 4 and f_score >= 4 and m_score >= 4
                then 'Champions'
            when f_score >= 3 and m_score >= 3
                then 'Loyal Customers'
            when r_score >= 3 and f_score <= 2
                then 'Potential Loyalists'
            when r_score >= 4 and f_score = 1
                then 'New Customers'
            when r_score <= 2 and f_score >= 3
                then 'At Risk'
            when r_score <= 2 and f_score <= 2 and m_score >= 2
                then 'Hibernating'
            else 'Lost'
        end as rfm_segment
    from rfm_scores
)

select * from rfm_segmented
order by rfm_score desc
"""

with open(f"{DBT_PATH}/models/marts/mart_rfm.sql", 'w') as f:
    f.write(mart_rfm_sql)
print(' mart_rfm.sql écrit')

 mart_rfm.sql écrit


## Cellule 7 — Écrire mart_clv.sql
> CLV simplifiée = Panier moyen × Fréquence annuelle × Durée relation estimée

In [7]:
mart_clv_sql = """
-- ============================================================
-- mart_clv : Customer Lifetime Value simplifiée — ST2
-- CLV = Panier moyen × Fréquence annuelle × Durée relation
-- 1 ligne = 1 client avec CLV et comparaison CAC estimé
-- ============================================================

with base as (
    -- Agrégation par client
    select
        t.customer_id,
        -- Panier moyen par commande
        round(sum(t.line_revenue) / count(distinct t.invoice_no), 2)    as avg_basket,
        -- Nb commandes totales
        count(distinct t.invoice_no)                                     as total_orders,
        -- CA total
        round(sum(t.line_revenue), 2)                                    as total_revenue,
        -- Durée relation en mois
        datediff('month',
            min(t.invoice_date),
            max(t.invoice_date)
        ) + 1                                                            as relation_months,
        -- Première et dernière commande
        min(t.invoice_date)                                              as first_order_date,
        max(t.invoice_date)                                              as last_order_date
    from fact_transactions t
    group by t.customer_id
),

clv_computed as (
    select
        b.customer_id,
        b.avg_basket,
        b.total_orders,
        b.total_revenue,
        b.relation_months,
        b.first_order_date,
        b.last_order_date,
        -- Fréquence annuelle estimée
        round(b.total_orders / greatest(b.relation_months / 12.0, 1), 2)    as annual_frequency,
        -- Durée relation estimée (en années) — cap à 3 ans
        least(round(b.relation_months / 12.0, 2), 3.0)                      as relation_years,
        -- CLV simplifiée
        round(
            b.avg_basket
            * round(b.total_orders / greatest(b.relation_months / 12.0, 1), 2)
            * least(round(b.relation_months / 12.0, 2), 3.0),
        2)                                                                   as clv_simple,
        -- Coût rétention estimé (hypothèse : 10% du CA)
        round(b.total_revenue * 0.10, 2)                                     as retention_cost_est,
        -- CLV nette
        round(
            b.avg_basket
            * round(b.total_orders / greatest(b.relation_months / 12.0, 1), 2)
            * least(round(b.relation_months / 12.0, 2), 3.0)
            - (b.total_revenue * 0.10),
        2)                                                                   as clv_net
    from base b
),

with_segment as (
    -- Jointure avec le segment RFM
    select
        c.*,
        r.rfm_segment,
        r.rfm_score,
        r.r_score,
        r.f_score,
        r.m_score,
        r.recency_days,
        r.monetary,
        r.country,
        -- Catégorie CLV
        case
            when c.clv_simple >= 1000 then 'High Value'
            when c.clv_simple >= 300  then 'Medium Value'
            else                           'Low Value'
        end as clv_category
    from clv_computed c
    left join mart_rfm r using (customer_id)
)

select * from with_segment
order by clv_simple desc
"""

with open(f"{DBT_PATH}/models/marts/mart_clv.sql", 'w') as f:
    f.write(mart_clv_sql)
print('mart_clv.sql écrit')

mart_clv.sql écrit


## Cellule 8 — Écrire mart_cohorts.sql
> Rétention mensuelle par cohorte d'acquisition (heatmap 12×12)

In [8]:
mart_cohorts = """
-- ============================================================
-- mart_cohorts : Rétention par cohorte d'acquisition — ST2
-- Cohorte = mois du premier achat
-- 1 ligne = 1 cohorte × 1 mois_ordre avec nb clients actifs
-- ============================================================

with first_purchase as (
    -- Mois du premier achat par client (cohorte)
    select
        customer_id,
        date_trunc('month', min(invoice_date))   as cohort_month,
        min(invoice_date)                        as first_order_date
    from fact_transactions
    group by customer_id
),

orders_with_cohort as (
    -- Joindre chaque commande à la cohorte du client
    select
        t.customer_id,
        f.cohort_month,
        date_trunc('month', t.invoice_date)       as order_month,
        -- Indice de période (0 = mois d'acquisition)
        datediff('month', f.cohort_month, date_trunc('month', t.invoice_date)) as period_number
    from fact_transactions t
    inner join first_purchase f using (customer_id)
),

cohort_size as (
    -- Taille de chaque cohorte (nb clients au mois 0)
    select
        cohort_month,
        count(distinct customer_id) as cohort_size
    from first_purchase
    group by cohort_month
),

retention_raw as (
    -- Nb de clients actifs par cohorte × période
    select
        cohort_month,
        period_number,
        count(distinct customer_id) as active_customers
    from orders_with_cohort
    group by cohort_month, period_number
)

select
    r.cohort_month,
    strftime(r.cohort_month, '%Y-%m')               as cohort_label,
    r.period_number,
    r.active_customers,
    s.cohort_size,
    -- Taux de rétention en %
    round(100.0 * r.active_customers / s.cohort_size, 1)  as retention_rate
from retention_raw r
inner join cohort_size s using (cohort_month)
order by r.cohort_month, r.period_number
"""

with open(f"{DBT_PATH}/models/marts/mart_cohorts.sql", 'w') as f:
    f.write(mart_cohorts)
print('mart_cohorts.sql écrit')

mart_cohorts.sql écrit


## Cellule 9 — Mettre à jour schema.yml (ajouter tests ST2)

In [9]:
# On lit l'existant ST1 et on ajoute les modèles ST2
schema_st2_addition = """

  # ── ST2 : RFM + CLV + Cohortes ────────────────────────────

  - name: mart_rfm
    description: "Segmentation RFM — 1 ligne = 1 client avec scores R,F,M et segment nommé"
    columns:
      - name: customer_id
        tests: [not_null, unique]
      - name: r_score
        tests:
          - not_null
          - accepted_values:
              values: [1, 2, 3, 4, 5]
      - name: f_score
        tests:
          - not_null
          - accepted_values:
              values: [1, 2, 3, 4, 5]
      - name: m_score
        tests:
          - not_null
          - accepted_values:
              values: [1, 2, 3, 4, 5]
      - name: rfm_segment
        tests:
          - not_null
          - accepted_values:
              values:
                - Champions
                - Loyal Customers
                - Potential Loyalists
                - New Customers
                - At Risk
                - Hibernating
                - Lost
      - name: monetary
        tests: [not_null]

  - name: mart_clv
    description: "CLV simplifiée par client — Panier moyen x Fréquence annuelle x Durée relation"
    columns:
      - name: customer_id
        tests: [not_null, unique]
      - name: clv_simple
        tests: [not_null]
      - name: rfm_segment
        tests: [not_null]
      - name: clv_category
        tests:
          - accepted_values:
              values: [High Value, Medium Value, Low Value]

  - name: mart_cohorts
    description: "Rétention mensuelle par cohorte d'acquisition — heatmap 12x12"
    columns:
      - name: cohort_month
        tests: [not_null]
      - name: period_number
        tests: [not_null]
      - name: retention_rate
        tests: [not_null]
"""

# Lire le schema.yml existant (ST1) et ajouter ST2
schema_path = f"{DBT_PATH}/models/schema.yml"

if os.path.exists(schema_path):
    with open(schema_path, 'r') as f:
        existing = f.read()
    # Eviter les doublons
    if 'mart_rfm' not in existing:
        with open(schema_path, 'a') as f:
            f.write(schema_st2_addition)
        print('schema.yml enrichi avec les tests ST2')
    else:
        print('mart_rfm déjà présent dans schema.yml — pas de doublon')
else:
    # Créer depuis zéro avec ST1 + ST2
    schema_full = """version: 2

models:
  - name: fact_transactions
    description: "Table de faits centrale — 1 ligne = 1 ligne de vente nette"
    columns:
      - name: invoice_no
        tests: [not_null]
      - name: customer_id
        tests: [not_null]
""" + schema_st2_addition
    with open(schema_path, 'w') as f:
        f.write(schema_full)
    print('✅ schema.yml créé avec ST1 + ST2')

schema.yml enrichi avec les tests ST2


## Cellule 10 — Lancer dbt run (uniquement modèles ST2)

In [10]:
import os
os.chdir(DBT_PATH)
print('Dossier courant :', os.getcwd())

print('\n=== dbt debug ===')
!dbt debug --profiles-dir ~/.dbt

Dossier courant : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/DBT_Gold

=== dbt debug ===
11:50:46  Running with dbt=1.11.11
11:50:46  dbt version: 1.11.11
11:50:46  python version: 3.12.13
11:50:46  python path: /usr/bin/python3
11:50:46  os info: Linux-6.6.122+-x86_64-with-glibc2.35
11:50:46  Using profiles dir at /root/.dbt
11:50:46  Using profiles.yml file at /root/.dbt/profiles.yml
11:50:46  Using dbt_project.yml file at /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/DBT_Gold/dbt_project.yml
11:50:46  adapter type: duckdb
11:50:46  adapter version: 1.10.1
11:50:47  Configuration:
11:50:47    profiles.yml file [OK found and valid]
11:50:47    dbt_project.yml file [OK found and valid]
11:50:47  Required dependencies:
11:50:47   - git [OK found]

11:50:47  Connection:
11:50:47    database: gold
11:50:47    schema: main
11:50:47    path: /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb
11:50:47    config_options: None
11:50:47    extensions: None
11:50:47    

In [11]:
# On lance uniquement les 3 nouveaux modèles ST2
print('\n=== dbt run — ST2 uniquement ===')
!dbt run --select mart_rfm mart_clv mart_cohorts --profiles-dir ~/.dbt


=== dbt run — ST2 uniquement ===
11:50:54  Running with dbt=1.11.11
11:50:55  Registered adapter: duckdb=1.10.1
11:50:59  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on 'mart_rfm' in
package 'ecommerce' (models/schema.yml). Arguments to generic tests should be
nested under the `arguments` property.
11:51:01  Found 12 models, 35 data tests, 485 macros
11:51:01  
11:51:01  Concurrency: 4 threads (target='dev')
11:51:01  
11:51:01  1 of 3 START sql table model main.mart_clv ..................................... [RUN]
11:51:01  2 of 3 START sql table model main.mart_cohorts ................................. [RUN]
11:51:01  3 of 3 START sql table model main.mart_rfm ..................................... [RUN]
11:51:03  1 of 3 OK created sql table model main.mart_clv ................................ [OK in 2.15s]
11:51:03  3 of 3 OK created sql table model main.mart_rfm ....................

In [12]:
print('\n=== dbt test — ST2 uniquement ===')
!dbt test --select mart_rfm mart_clv mart_cohorts --profiles-dir ~/.dbt


=== dbt test — ST2 uniquement ===
11:51:12  Running with dbt=1.11.11
11:51:13  Registered adapter: duckdb=1.10.1
11:51:14  Found 12 models, 35 data tests, 485 macros
11:51:14  
11:51:14  Concurrency: 4 threads (target='dev')
11:51:14  
11:51:14  1 of 19 START test accepted_values_mart_clv_clv_category__High_Value__Medium_Value__Low_Value  [RUN]
11:51:14  2 of 19 START test accepted_values_mart_rfm_f_score__1__2__3__4__5 ............. [RUN]
11:51:14  3 of 19 START test accepted_values_mart_rfm_m_score__1__2__3__4__5 ............. [RUN]
11:51:14  4 of 19 START test accepted_values_mart_rfm_r_score__1__2__3__4__5 ............. [RUN]
11:51:17  2 of 19 PASS accepted_values_mart_rfm_f_score__1__2__3__4__5 ................... [PASS in 3.38s]
11:51:17  1 of 19 PASS accepted_values_mart_clv_clv_category__High_Value__Medium_Value__Low_Value  [PASS in 3.39s]
11:51:17  4 of 19 PASS accepted_values_mart_rfm_r_score__1__2__3__4__5 ................... [PASS in 3.39s]
11:51:17  5 of 19 START test acc

## Cellule 11 — Vérification KPIs ST2 depuis DuckDB

In [13]:
import duckdb
import pandas as pd

con = duckdb.connect(GOLD_DB)

print('=== Tables disponibles ===')
print(con.execute('SHOW TABLES').df().to_string())

=== Tables disponibles ===
                 name
0        dim_customer
1            dim_date
2         dim_payment
3        fact_returns
4   fact_transactions
5            mart_clv
6        mart_cohorts
7         mart_funnel
8            mart_rfm
9         stg_returns
10       stg_shipping
11   stg_transactions


In [14]:
print('\n=== KPIs RFM — Distribution des segments ===')
rfm_dist = con.execute("""
    select
        rfm_segment,
        count(*)                                    as nb_clients,
        round(100.0*count(*)/sum(count(*)) over(),1) as pct_clients,
        round(sum(monetary),0)                      as ca_total_gbp,
        round(100.0*sum(monetary)/sum(sum(monetary)) over(),1) as pct_ca,
        round(avg(monetary),0)                      as ca_moyen_client,
        round(avg(frequency),1)                     as freq_moyenne,
        round(avg(recency_days),0)                  as recence_moy_jours
    from mart_rfm
    group by rfm_segment
    order by ca_total_gbp desc
""").df()
print(rfm_dist.to_string(index=False))


=== KPIs RFM — Distribution des segments ===
        rfm_segment  nb_clients  pct_clients  ca_total_gbp  pct_ca  ca_moyen_client  freq_moyenne  recence_moy_jours
          Champions         987         23.2     3463580.0    61.8           3509.0          10.1               13.0
    Loyal Customers        1187         27.9     1470767.0    26.2           1239.0           3.9               67.0
        Hibernating         668         15.7      335348.0     6.0            502.0           1.3              219.0
Potential Loyalists         530         12.5      166992.0     3.0            315.0           1.0               38.0
               Lost         765         18.0      137105.0     2.4            179.0           1.5              160.0
            At Risk         116          2.7       32015.0     0.6            276.0           2.7              134.0


In [15]:
print('\n=== KPIs CLV — Par segment ===')
clv_seg = con.execute("""
    select
        rfm_segment,
        count(*)                       as nb_clients,
        round(avg(clv_simple),0)       as clv_moy_gbp,
        round(min(clv_simple),0)       as clv_min,
        round(max(clv_simple),0)       as clv_max,
        round(avg(clv_net),0)          as clv_net_moy,
        round(avg(retention_cost_est),0) as cout_retention_moy
    from mart_clv
    group by rfm_segment
    order by clv_moy_gbp desc
""").df()
print(clv_seg.to_string(index=False))


=== KPIs CLV — Par segment ===
        rfm_segment  nb_clients  clv_moy_gbp  clv_min  clv_max  clv_net_moy  cout_retention_moy
          Champions         986       3110.0     90.0 112570.0       2759.0               351.0
    Loyal Customers        1187        766.0     34.0  12038.0        642.0               124.0
            At Risk         117        120.0     11.0    349.0         92.0                28.0
        Hibernating         670         71.0     17.0    989.0         21.0                50.0
               Lost         766         56.0      0.0    418.0         38.0                18.0
Potential Loyalists         527         25.0      0.0    405.0         -6.0                32.0


In [16]:
print('\n=== Segment At Risk — CA en danger ===')
at_risk = con.execute("""
    select
        count(*)                      as nb_clients_at_risk,
        round(sum(total_revenue),0)   as ca_en_danger_gbp,   -- ← total_revenue au lieu de monetary
        round(avg(recency_days),0)    as recence_moy_jours,
        round(avg(clv_simple),0)      as clv_moy_at_risk
    from mart_clv
    where rfm_segment = 'At Risk'
""").df()
print(at_risk.to_string(index=False))


=== Segment At Risk — CA en danger ===
 nb_clients_at_risk  ca_en_danger_gbp  recence_moy_jours  clv_moy_at_risk
                117           32271.0              134.0            120.0


In [17]:
print('\n=== Top 10 Champions (liste opérationnelle CRM) ===')
champions = con.execute("""
    select
        customer_id,
        country,
        recency_days,
        total_orders      as nb_commandes,    -- ← frequency → total_orders
        total_revenue     as ca_total_gbp,    -- ← monetary  → total_revenue
        clv_simple        as clv_gbp,
        rfm_score
    from mart_clv
    where rfm_segment = 'Champions'
    order by clv_simple desc
    limit 10
""").df()
print(champions.to_string(index=False))


=== Top 10 Champions (liste opérationnelle CRM) ===
customer_id        country  recency_days  nb_commandes  ca_total_gbp   clv_gbp  rfm_score
      14911           EIRE             2           196     112919.11 112570.16        5.0
      13089 United Kingdom             3            92      48144.76  47994.64        5.0
      17841 United Kingdom             2           123      39637.53  39516.55        5.0
      14088 United Kingdom            11            13      39065.48  35940.28        5.0
      14646    Netherlands             3            56      35465.68  35355.22        5.0
      14298 United Kingdom             9            40      34183.45  34075.58        5.0
      14156           EIRE            10            49      33139.39  33139.19        5.0
      12748 United Kingdom             1           203      25740.32  25660.57        5.0
      15311 United Kingdom             1            91      24025.47  23951.89        5.0
      13081 United Kingdom            12       

In [18]:
print('\n=== Rétention cohortes — Mois 0, 1, 2, 3 ===')
cohorts = con.execute("""
    select
        cohort_label,
        cohort_size,
        max(case when period_number=0 then retention_rate end) as mois_0,
        max(case when period_number=1 then retention_rate end) as mois_1,
        max(case when period_number=2 then retention_rate end) as mois_2,
        max(case when period_number=3 then retention_rate end) as mois_3
    from mart_cohorts
    group by cohort_label, cohort_size
    order by cohort_label
""").df()
print(cohorts.to_string(index=False))

con.close()


=== Rétention cohortes — Mois 0, 1, 2, 3 ===
cohort_label  cohort_size  mois_0  mois_1  mois_2  mois_3
     2010-12          852   100.0    36.2    31.7    37.1
     2011-01          406   100.0    21.2    26.6    22.2
     2011-02          367   100.0    18.0    18.8    28.3
     2011-03          450   100.0    14.9    24.7    19.8
     2011-04          292   100.0    21.2    20.2    20.5
     2011-05          279   100.0    19.4    16.8    17.2
     2011-06          235   100.0    17.0    14.9    26.4
     2011-07          192   100.0    16.1    21.9    22.9
     2011-08          165   100.0    18.8    24.2    24.8
     2011-09          294   100.0    23.5    30.6    11.9
     2011-10          356   100.0    23.6    11.5     NaN
     2011-11          319   100.0    10.7     NaN     NaN
     2011-12           38   100.0     NaN     NaN     NaN


## Cellule 12 — Générer la doc dbt (lineage DAG)

In [19]:
os.chdir(DBT_PATH)
!dbt docs generate --profiles-dir ~/.dbt
print('\n Doc générée dans DBT_Gold/target/')
print('Fichiers :', os.listdir(f"{DBT_PATH}/target"))

11:51:36  Running with dbt=1.11.11
11:51:37  Registered adapter: duckdb=1.10.1
11:51:38  Found 12 models, 35 data tests, 485 macros
11:51:38  
11:51:38  Concurrency: 4 threads (target='dev')
11:51:38  
11:51:48  Building catalog
11:51:48  Catalog written to /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/DBT_Gold/target/catalog.json

 Doc générée dans DBT_Gold/target/
Fichiers : ['catalog.json', 'index.html', 'run', 'compiled', 'partial_parse.msgpack', 'semantic_manifest.json', 'run_results.json', 'manifest.json', 'graph_summary.json', 'graph.gpickle']


---
##  Récapitulatif — Tables Gold ST2 créées

| Table | Type | ST | Description |
|---|---|---|---|
| `mart_rfm` | Table | ST2 | Scores R,F,M (1→5) + segment nommé par client |
| `mart_clv` | Table | ST2 | CLV simplifiée + CLV nette + catégorie valeur |
| `mart_cohorts` | Table | ST2 | Taux rétention % par cohorte × période |

### Tables ST1 réutilisées par ST2
| Table | Utilisée dans |
|---|---|
| `fact_transactions` | Base de calcul RFM et CLV |
| `dim_customer` | Enrichissement pays |
| `dim_date` | Calendrier cohortes |

**Étape suivante :** Notebook EDA ST2 — Treemap segments, scatter R×F, heatmap cohortes, box plot CLV.

In [20]:
# ══════════════════════════════════════════════════════════════
# Cellule 13 — Export de toutes les tables Gold en CSV
# ══════════════════════════════════════════════════════════════
import duckdb
import os

# Dossier de sortie CSV
CSV_PATH = f"{DATA_PATH}/gold/csv_exports"
os.makedirs(CSV_PATH, exist_ok=True)

con = duckdb.connect(GOLD_DB)

# Récupérer la liste de toutes les tables
tables = con.execute("SHOW TABLES").df()["name"].tolist()
print(f"Tables trouvées : {tables}\n")

# Exporter chaque table en CSV
for table in tables:
    output_file = f"{CSV_PATH}/{table}.csv"
    con.execute(f"""
        COPY {table} TO '{output_file}'
        (HEADER, DELIMITER ',')
    """)
    row_count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f" {table}.csv exporté ({row_count:,} lignes) → {output_file}")

con.close()
print(f"\n📁 Tous les CSV sont dans : {CSV_PATH}")
print(f"Fichiers : {os.listdir(CSV_PATH)}")

Tables trouvées : ['dim_customer', 'dim_date', 'dim_payment', 'fact_returns', 'fact_transactions', 'mart_clv', 'mart_cohorts', 'mart_funnel', 'mart_rfm', 'stg_returns', 'stg_shipping', 'stg_transactions']

 dim_customer.csv exporté (4,253 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/dim_customer.csv
 dim_date.csv exporté (396 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/dim_date.csv
 dim_payment.csv exporté (3 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/dim_payment.csv
 fact_returns.csv exporté (8,872 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/fact_returns.csv
 fact_transactions.csv exporté (372,684 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/fact_transactions.csv
 mart_clv.csv exporté (4,253 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/mart_clv.csv
 mart_cohorts.csv exporté 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 stg_transactions.csv exporté (372,684 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/stg_transactions.csv

📁 Tous les CSV sont dans : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports
Fichiers : ['dim_customer.csv', 'dim_date.csv', 'dim_payment.csv', 'fact_returns.csv', 'fact_transactions.csv', 'mart_clv.csv', 'mart_cohorts.csv', 'mart_funnel.csv', 'mart_rfm.csv', 'stg_returns.csv', 'stg_shipping.csv', 'stg_transactions.csv']
